In [1]:
from mini_competition.data_functions import load_data, add_zero_sales, split_train_set, write_result, PROCCESED_PATH, EXTRACTED_PATH, PROCCESED_PATH
import pandas as pd
import numpy as np

data = load_data(PROCCESED_PATH / "train.parquet" )
# data = data[data["date"] > "2015-08-16"]

data.head()

,date,store_nbr,item_nbr,unit_sales,onpromotion
0,2013-01-01,25,103665,7.0,None
1,2013-01-01,25,105574,1.0,None
2,2013-01-01,25,105575,2.0,None
3,2013-01-01,25,108079,1.0,None
4,2013-01-01,25,108701,1.0,None


In [ ]:
# add zero sales lines
data = add_zero_sales(data)

data.info()

In [ ]:
item_data = load_data(EXTRACTED_PATH / "items.csv")
store_data = load_data(EXTRACTED_PATH / "stores.csv")
oil_data = load_data(EXTRACTED_PATH / "oil.csv" )
holiday_data = load_data(EXTRACTED_PATH / "holidays_events.csv" )

def add_item_data(d : pd.DataFrame, items : pd.DataFrame) -> pd.DataFrame:
    return  d.merge(items, on="item_nbr", how="left").copy()

def add_store_data(d :pd.DataFrame, stores: pd.DataFrame) -> pd.DataFrame:
    return  d.merge(stores, on="store_nbr", how="left").copy()

def add_holidays(d: pd.DataFrame, holidays: pd.DataFrame) -> pd.DataFrame:
    h = holidays.copy()
    h["date"] = pd.to_datetime(h["date"])
    national = h[(h["locale"] == "National") & (~h["transferred"])]["date"]

    one_day = pd.Timedelta(days=1)
    d = d.copy()
    d["date"] = pd.to_datetime(d["date"])
    d["is_national_holiday"] = d["date"].isin(national).astype("float32")
    d["holiday_tomorrow"]    = (d["date"] + one_day).isin(national).astype("float32")
    d["holiday_yesterday"]   = (d["date"] - one_day).isin(national).astype("float32")
    return d

def add_oil_lag_features(df: pd.DataFrame, oil: pd.DataFrame) -> pd.DataFrame:
    oil = oil.copy()
    oil["date"] = pd.to_datetime(oil["date"])
    s = oil.set_index("date")["dcoilwtico"]

    # Oil has no weekends/holidays -> make it daily, carry last known price forward
    days = pd.date_range(s.index.min(), df["date"].max(), freq="D")
    s = s.reindex(days).ffill()

    shifted = s.shift(16)
    feats = pd.DataFrame({
        "date": days,
        "oil_lag_16": shifted.values,
        "oil_rolling_mean_7": shifted.rolling(7).mean().values,
        "oil_rolling_mean_28": shifted.rolling(28).mean().values,
    })

    return df.merge(feats, on="date", how="left")

# add lag features, unit_sales and promo rolling average
def add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["store_nbr", "item_nbr", "date"]).reset_index(drop=True)
    df["log_sales"] = np.log1p(df["unit_sales"].astype("float32"))
    df["onpromotion"] = df["onpromotion"].fillna(False).astype("int8")

    keys = ["store_nbr", "item_nbr"]
    g = df.groupby(keys)["log_sales"]
    gp = df.groupby(keys)["onpromotion"]

    for lag in [16, 21, 28, 35]:
        df[f"lag_{lag}"] = g.shift(lag)

    df["rolling_mean_7"]  = g.transform(lambda s: s.shift(16).rolling(7).mean())
    df["rolling_mean_28"] = g.transform(lambda s: s.shift(16).rolling(28).mean())

    # number of promo days for this store-item over the previous 16 days (today excluded)
    df["promo_count_16d"] = gp.transform(
        lambda s: s.shift(1).rolling(16, min_periods=1).sum()
    ).fillna(0).astype("int8")

    return df.drop(columns=["log_sales"])

def log(df: pd.DataFrame, msg: str = "") -> pd.DataFrame:
    print(f"{msg}: {df.shape[0]:,} rows × {df.shape[1]} cols")
    return df

data_plus = (
    data
    .pipe(add_item_data, item_data)
    .pipe(log, "items added")
    .pipe(add_store_data, store_data)
    .pipe(log, "stores added")
    .pipe(add_holidays, holiday_data)
    .pipe(log, "holidays added")
    .pipe(add_oil_lag_features, oil_data)
    .pipe(log, "oil added")
    .pipe(add_lag_features)
    .pipe(log, "lags added")
)

# drop none rows
cols = ["lag_16", "lag_21","lag_28", "lag_35","rolling_mean_7", "rolling_mean_28"]
data_plus = data_plus.dropna(subset=cols, how="any")

data_plus.head()

items added: 158,663,232 rows × 8 cols
stores added: 158,663,232 rows × 12 cols
holidays added: 158,663,232 rows × 15 cols
oil added: 158,663,232 rows × 18 cols
lags added: 158,663,232 rows × 25 cols


,date,store_nbr,item_nbr,unit_sales,onpromotion,family,class,perishable,city,state,...,oil_lag_16,oil_rolling_mean_7,oil_rolling_mean_28,lag_16,lag_21,lag_28,lag_35,rolling_mean_7,rolling_mean_28,promo_count_16d
43,2015-09-29,1,96995,0.0,0,GROCERY I,1093,0,Quito,Pichincha,...,44.75,45.167143,43.702857,0.0,0.0,0.0,0.0,0.0,0.0,0
44,2015-09-30,1,96995,0.0,0,GROCERY I,1093,0,Quito,Pichincha,...,44.07,44.888571,43.779286,0.0,0.0,0.0,0.0,0.0,0.0,0
45,2015-10-01,1,96995,0.0,0,GROCERY I,1093,0,Quito,Pichincha,...,44.58,44.697143,43.850714,0.0,0.0,0.0,0.0,0.0,0.0,0
46,2015-10-02,1,96995,0.0,0,GROCERY I,1093,0,Quito,Pichincha,...,47.12,45.124286,44.078214,0.0,0.0,0.0,0.0,0.0,0.0,0
47,2015-10-03,1,96995,0.0,0,GROCERY I,1093,0,Quito,Pichincha,...,46.93,45.278571,44.290000,0.0,0.0,0.0,0.0,0.0,0.0,0


In [ ]:
data_plus.info()

<class 'pandas.DataFrame'>
Index: 149291640 entries, 43 to 158663231
Data columns (total 25 columns):
 #   Column               Dtype         
---  ------               -----         
 0   date                 datetime64[us]
 1   store_nbr            int8          
 2   item_nbr             int32         
 3   unit_sales           float32       
 4   onpromotion          int8          
 5   family               str           
 6   class                int64         
 7   perishable           int64         
 8   city                 str           
 9   state                str           
 10  type                 str           
 11  cluster              int64         
 12  is_national_holiday  float32       
 13  holiday_tomorrow     float32       
 14  holiday_yesterday    float32       
 15  oil_lag_16           float64       
 16  oil_rolling_mean_7   float64       
 17  oil_rolling_mean_28  float64       
 18  lag_16               float32       
 19  lag_21               float32    

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import TargetEncoder, FunctionTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
import numpy as np
from scipy import stats

CAT_COLS = ["store_nbr", "family", "city", "state", "type", "cluster", "weekday_num"]

def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["onpromotion"] = df["onpromotion"].fillna(False).astype("int8")
    df["perishable"] = df["perishable"].astype("int8")

    df["date"] = pd.to_datetime(df["date"])
    df["weekday_num"] = df["date"].dt.day_of_week.astype("category")
    df["monthday_num"] = df["date"].dt.day.astype("int8")
    df["month_num"]    = df["date"].dt.month.astype("int8")
    df["days_since_payday"] = (df["date"].dt.day % 15).astype("int8")

    df["is_national_holiday"] = df["is_national_holiday"].fillna(0).astype("int8")
    df["holiday_tomorrow"] = df["holiday_tomorrow"].fillna(0).astype("int8")
    df["holiday_yesterday"] = df["holiday_yesterday"].fillna(0).astype("int8")

    df[CAT_COLS] = df[CAT_COLS].astype("category")
    return df.drop(columns="date")

df = data_plus

(x_train, y_train, x_valid, y_valid) = split_train_set(df, "2017-08-16", train_samples=5_000_000, validation_samples=0)

# Target-encode item_nbr and pass the other columns through unchanged
preprocess = ColumnTransformer(
    [("item_te", TargetEncoder(), ["item_nbr", "class"])],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")   # keeps store_nbr as 'category'

model = TransformedTargetRegressor(
    regressor=make_pipeline(
        FunctionTransformer(prepare_features),
        preprocess,
        HistGradientBoostingRegressor(
            categorical_features=CAT_COLS,
            max_iter=1000, learning_rate=0.05, max_leaf_nodes=127,
            early_stopping=True,
        ),
    ),
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False,   # clipping can't be undone, so skip the round-trip check
)


model.fit(x_train, y_train)

if len(x_valid):
    pred = model.predict(x_valid)
    stats.describe(pred)
else:
    pred = None

ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.

In [ ]:
# scoring
if pred != None:
    from sklearn.metrics import root_mean_squared_log_error

    error = root_mean_squared_log_error(y_valid, np.clip(pred, 0, None))
    print(f"Error for random sample set is: {error}")

Error for random sample set is: 0.5509545989779921


In [ ]:
# load test set and add lag
from pathlib import Path
from datetime import timedelta

test_data = load_data(EXTRACTED_PATH / "test.csv", dtype={"id": int})
test_data["unit_sales"] = None

# figure out what training data we need to include to calculate the lags
min_test_date = pd.to_datetime(test_data["date"]).min()
train_split_date = min_test_date - timedelta(days=16+30)

print(f"test set starts from date: {min_test_date}, attaching dates > {train_split_date}")

# combine with previous history
combined = pd.concat([data[data["date"] >= train_split_date], test_data], ignore_index=True)

# add zero sales
combined_zeros = add_zero_sales(combined)

combined_zeros.head()

# add aditonial features
combined_plus = (
    combined_zeros
    .pipe(add_item_data, item_data)
    .pipe(log, "items added")
    .pipe(add_store_data, store_data)
    .pipe(log, "stores added")
    .pipe(add_holidays, holiday_data)
    .pipe(log, "holidays added")
    .pipe(add_oil_lag_features, oil_data)
    .pipe(log, "oil added")
    .pipe(add_lag_features)
    .pipe(log, "lags added")
)

# extract only the testset
test_set = combined_plus[combined_plus["id"] > 0]

test_set.head()

test set starts from date: 2017-08-16 00:00:00, attaching dates > 2017-07-01 00:00:00
items added: 13,713,408 rows × 9 cols
stores added: 13,713,408 rows × 13 cols
holidays added: 13,713,408 rows × 16 cols
oil added: 13,713,408 rows × 19 cols
lags added: 13,713,408 rows × 26 cols


,date,store_nbr,item_nbr,unit_sales,onpromotion,id,family,class,perishable,city,...,oil_lag_16,oil_rolling_mean_7,oil_rolling_mean_28,lag_16,lag_21,lag_28,lag_35,rolling_mean_7,rolling_mean_28,promo_count_16d
46,2017-08-16,1,96995,0,0,125497040.0,GROCERY I,1093,0,Quito,...,50.21,49.252857,46.591429,1.098612,0.0,0.000000,0.0,0.156945,0.127983,0
47,2017-08-17,1,96995,0,0,125707694.0,GROCERY I,1093,0,Quito,...,49.19,49.455714,46.704643,0.000000,0.0,0.000000,0.0,0.156945,0.127983,0
48,2017-08-18,1,96995,0,0,125918348.0,GROCERY I,1093,0,Quito,...,49.60,49.601429,46.865000,0.693147,0.0,1.386294,0.0,0.255966,0.152738,0
49,2017-08-19,1,96995,0,0,126129002.0,GROCERY I,1093,0,Quito,...,49.03,49.598571,46.990357,1.098612,0.0,1.098612,0.0,0.412910,0.191974,0
50,2017-08-20,1,96995,0,0,126339656.0,GROCERY I,1093,0,Quito,...,49.57,49.577143,47.180357,0.000000,0.0,0.000000,0.0,0.412910,0.191974,0


In [ ]:
# create csv
import os
from pathlib import Path

EXTRACTED_FOLDER = Path("./../data/raw") / "extracted"
RESULT_FOLDER = Path("./../data/results")

# make predictions
features = test_set.columns.drop(["unit_sales", "id"])
test_pred = model.predict(test_set[features])

# clip to 0 as minimum
test_pred_positive = np.clip(test_pred, 0, None)

submission_data = pd.DataFrame({
    "id": test_set["id"].to_numpy().astype("int64"),
    "unit_sales": np.asarray(test_pred_positive, dtype=float).ravel(),
})

write_result(submission_data, "lag_features_plus_result_v4.csv")



# Scores

* test_split: "2017-07-31", data from ""2016-07-01" onwards, samples=2_000_000 **private: 0.54674 public: 0.57927**

* test_split: "2017-07-31", data from ""2016-07-01" onwards, samples=2_000_000, adds promotion lag and days since payday feature **private: 0.55707 public: 0.58556**



